# 11 · Model interpretation

Explains the two selected models:
- **Poverty vulnerability:** LightGBM (`models/poverty_vulnerability_pipeline.joblib`).
- **VUP coverage gap:** logistic regression (`models/vup_gap_pipeline.joblib`).

| | |
|---|---|
| **Explanations** | SHAP values in log-odds, summed back to the original features (one-hot levels and missing-value indicators included), so they stay additive. The notebook checks that base value + SHAP values = the model's log-odds for every household. |
| **Data explained** | Poverty model: the final holdout (3,022 households, never used in training). VUP model: all 3,079 high-need households, because the holdout has only 622. Dependence plots describe the model, not its accuracy. |
| **Outputs** | `reports/interpretation/*.png`, `reports/interpretation/*.csv`, `docs/model_card_draft.md` (written separately) |

**How to read it.** SHAP shows what drives the *model's* predictions. It is not a causal effect.
- **Correlated features split credit.** Several features are strongly correlated (notebook 08), so importance is also reported by feature group.
- **The coverage model describes current VUP coverage.** It says which profiles are reached, not who should be.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models" / "common.py").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.models import common as cm
from src.models import interpretation as it
from src.models import plots as pl
from src.models import poverty_model as pm
from src.models import vup_gap_model as vm

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
OUT = ROOT / "reports" / "interpretation"
OUT.mkdir(parents=True, exist_ok=True)

pov_pipe = joblib.load(pm.PIPELINE_PATH)
vup_pipe = joblib.load(vm.PIPELINE_PATH)
pov_spec, vup_spec = pm.FEATURE_SETS["full"], vm.FEATURES
df = cm.load_frame().merge(cm.load_splits(), on=["hhid", "clust"])
hold = df[df["split"].eq("holdout")].reset_index(drop=True)
hn = vm.load_population()
gap = pd.read_parquet(vm.GAP_SCORES_PATH)
hn = hn.merge(gap[["hhid", "predicted_vup_coverage"]], on="hhid")
print(f"Poverty model: {type(pov_pipe.named_steps['model']).__name__}, {len(pov_spec.columns)} features; "
      f"VUP model: {type(vup_pipe.named_steps['model']).__name__}, {len(vup_spec.columns)} features")

In [ ]:
pov_shap_t, pov_base = it.explain(pov_pipe, hold[pov_spec.columns])
pov_shap = it.aggregate(pov_shap_t, pov_spec)
vup_bg = hn[hn["split"].eq("development")][vup_spec.columns]
vup_shap_t, vup_base = it.explain(vup_pipe, hn[vup_spec.columns], background=vup_bg)
vup_shap = it.aggregate(vup_shap_t, vup_spec)
print(f"Poverty SHAP: {pov_shap.shape}, base log-odds {pov_base:.3f} (p = {1 / (1 + np.exp(-pov_base)):.3f}); additivity checked")
print(f"VUP SHAP: {vup_shap.shape}, base log-odds {vup_base:.3f} (p = {1 / (1 + np.exp(-vup_base)):.3f}); additivity checked")

## 1. Global importance

In [ ]:
imp = {}
for name, s, data in [("poverty", pov_shap, hold), ("vup", vup_shap, hn)]:
    imp[name] = it.global_importance(s, data["weight"])
    imp[name][0].rename("mean_abs_shap").to_frame().assign(group=lambda t: t.index.map(it.feature_group)).to_csv(OUT / f"shap_global_features_{name}.csv")
    imp[name][1].rename("mean_abs_shap").to_csv(OUT / f"shap_global_groups_{name}.csv")

def bar_panel(ax, s, color, n=15):
    s = s.head(n)[::-1]
    ax.barh(range(len(s)), s.to_numpy(), color=color, height=0.65)
    ax.set_yticks(range(len(s)), [it.pretty(i) for i in s.index], fontsize=8.5)
    for i, v in enumerate(s.to_numpy()):
        ax.text(v, i, f" {v:.2f}", va="center", fontsize=7.5, color=pl.INK["secondary"])
    ax.set_xlabel("Mean |SHAP|, log-odds (household-weighted)", fontsize=8.5, color=pl.INK["secondary"])

for name, title, color in [("poverty", "Poverty model (LightGBM): what drives predicted poverty risk", pl.SERIES[2]),
                           ("vup", "VUP coverage model (logistic): what drives predicted coverage of high-need households", pl.SERIES[0])]:
    fig, (a1, a2) = pl.figure(title, "Left: 15 most important features. Right: feature groups (|sum of the group's SHAP values|, so correlated features are pooled)",
                              width=12, height=6.2, ncols=2)
    bar_panel(a1, imp[name][0], color)
    bar_panel(a2, imp[name][1], color, n=12)
    path = pl.save(fig, OUT / f"shap_global_{name}.png", top=0.82, bottom=0.12, left=0.2, wspace=0.55)
    display(Image(str(path)))
pd.concat({"poverty: top features": imp["poverty"][0].head(10).reset_index(drop=False).rename(columns={"index": "feature", 0: "mean_abs_shap"}),
           "vup: top features": imp["vup"][0].head(10).reset_index(drop=False).rename(columns={"index": "feature", 0: "mean_abs_shap"})}, axis=1)

**Check: is the coverage model's top feature a hidden VUP signal?** The logistic model gives the largest single weight to the number of casual wage workers. Public Works is casual labour, so the next cell tests whether casual work proxies participation.

In [ ]:
pw = hn["vup_classic_pw"].eq(1) | hn["vup_expanded_pw"].eq(1)
other_vup = hn["vup_any"].eq(1) & ~pw
cas = hn["n_casual_wage_pf"].gt(0)
w_ = hn["weight"]
share = lambda m: np.average(cas[m], weights=w_[m])
print(f"Share of high-need households with a casual wage worker: Public Works {share(pw):.2f}, other VUP {share(other_vup):.2f}, no VUP {share(hn['vup_any'].eq(0)):.2f}")
print(f"VUP coverage with / without a casual wage worker: {np.average(hn.loc[cas, 'vup_any'], weights=w_[cas]):.3f} / {np.average(hn.loc[~cas, 'vup_any'], weights=w_[~cas]):.3f}")
coef = pd.Series(vup_pipe.named_steps["model"].coef_[0], index=vup_pipe.named_steps["prep"].get_feature_names_out())
print("Logistic coefficients (standardised) of the employment counts:")
display(coef[[c for c in coef.index if c.startswith("num__n_") and c.endswith("_pf")]].round(3).to_frame("coefficient"))

Casual work is as common among non-recipients as among Public Works households, and coverage is the same with or without a casual worker. So it is not a hidden VUP signal. Its large positive coefficient is offset by negative coefficients on the correlated counts of employed members and farm wage workers: a multicollinearity artefact. **For the logistic model, read importance at group level** (right-hand panel above), not feature by feature.

## 2. Dependence plots

Each panel plots the feature's SHAP value (log-odds; above 0 raises the predicted probability) against its value. The line shows binned means. Categorical features (income source, district) show the mean SHAP per category with its interquartile range.
- **Food share:** the VUP model has no expenditure shares, so its food panel uses the food price shock (`food_shock_pp`, correlation 0.89 with food share).
- **Education:** head's education, 0 none … 5 tertiary.
- **Assets:** the number of key assets.

In [ ]:
LIV = {"own_farming": "Own farming", "farm_wage": "Farm wage", "nonfarm_wage": "Non-farm wage",
       "nonfarm_self_employment": "Non-farm self-emp.", "no_employed_member": "No employed member"}

def numeric_panel(ax, x, s, label, color, discrete=False):
    ok = x.notna()
    x, s = x[ok].astype(float), s[ok]
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(x)) if discrete else 0
    ax.scatter(x + jitter, s, s=5, alpha=0.25, color=color, linewidths=0)
    if discrete:
        m = s.groupby(x).mean()
    else:
        bins = pd.qcut(x, 12, duplicates="drop")
        m = s.groupby(bins, observed=True).mean()
        m.index = x.groupby(bins, observed=True).median().to_numpy()
    ax.plot(m.index, m.to_numpy(), "-", color=pl.INK["primary"], linewidth=2)
    ax.axhline(0, color=pl.BASELINE, linewidth=1)
    ax.set_xlabel(label, fontsize=8.5, color=pl.INK["secondary"])
    ax.grid(axis="both", color=pl.GRID)

def categorical_panel(ax, cat, s, label, color, order=None, rename=None):
    g = s.groupby(cat)
    stats = pd.DataFrame({"mean": g.mean(), "q25": g.quantile(0.25), "q75": g.quantile(0.75), "n": g.size()})
    stats = stats.loc[order] if order is not None else stats.sort_values("mean")
    y = np.arange(len(stats))
    ax.hlines(y, stats["q25"], stats["q75"], color=color, linewidth=2, alpha=0.45)
    ax.plot(stats["mean"], y, "o", color=color, markersize=5)
    ax.axvline(0, color=pl.BASELINE, linewidth=1)
    labels = [(rename or {}).get(i, i) for i in stats.index]
    ax.set_yticks(y, labels, fontsize=7 if len(stats) > 10 else 8)
    ax.set_xlabel(f"SHAP (log-odds): {label}", fontsize=8.5, color=pl.INK["secondary"])
    ax.grid(axis="x", color=pl.GRID)
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))

def dependence_figure(name, shap_f, data, color, food_col, title):
    fig, axes = plt.subplots(2, 3, figsize=(13, 9))
    fig.patch.set_facecolor(pl.SURFACE)
    fig.text(0.03, 0.965, title, fontsize=14, fontweight="bold", color=pl.INK["primary"])
    fig.text(0.03, 0.935, "SHAP value (log-odds) against the feature; line = binned mean. Categorical: mean and interquartile range",
             fontsize=10, color=pl.INK["secondary"])
    fig.text(0.03, 0.01, pl.SOURCE, fontsize=8, color=pl.INK["muted"])
    for ax in axes.ravel():
        pl.style(ax, "both")
    numeric_panel(axes[0, 0], data["price_change_pct"], shap_f["price_change_pct"], "Household CPI shock, Jan 2024-Sep 2026, %", color)
    numeric_panel(axes[0, 1], data[food_col], shap_f[food_col], "Food share of consumption" if food_col == "share_food" else "Food price shock, pp of total", color)
    numeric_panel(axes[0, 2], data["head_education_code"], shap_f["head_education_code"], "Head's education (0 none ... 5 tertiary)", color, discrete=True)
    numeric_panel(axes[1, 0], data["n_key_assets"], shap_f["n_key_assets"], "Number of key assets", color, discrete=True)
    categorical_panel(axes[1, 1], data["main_livelihood_pf"], shap_f["main_livelihood_pf"], "income source", color, rename=LIV)
    categorical_panel(axes[1, 2], data["district_name"], shap_f["district"], "district", color)
    for ax in axes[:, 0]:
        ax.set_ylabel("SHAP, log-odds", fontsize=8.5, color=pl.INK["secondary"])
    fig.subplots_adjust(top=0.9, bottom=0.08, left=0.06, right=0.98, hspace=0.32, wspace=0.42)
    path = OUT / f"shap_dependence_{name}.png"
    fig.savefig(path, dpi=150, facecolor=pl.SURFACE)
    plt.close(fig)
    return path

display(Image(str(dependence_figure("poverty", pov_shap, hold, pl.SERIES[2], "share_food", "Poverty model: how six features move predicted poverty risk"))))
display(Image(str(dependence_figure("vup", vup_shap, hn, pl.SERIES[0], "food_shock_pp", "VUP coverage model: how six features move predicted coverage"))))

In [ ]:
def direction(shap_f, data, feat):
    x = data[feat].astype(float)
    ok = x.notna()
    return np.corrcoef(x[ok], shap_f.loc[ok, feat])[0, 1]

summary = pd.DataFrame({m: {f: direction(s, d, f) for f in ["price_change_pct", "head_education_code", "n_key_assets"]}
                        for m, s, d in [("poverty", pov_shap, hold), ("vup", vup_shap, hn)]})
summary.loc["food (share_food / food_shock_pp)"] = [direction(pov_shap, hold, "share_food"), direction(vup_shap, hn, "food_shock_pp")]
print("Correlation between feature value and its SHAP value (+ = higher value raises the predicted probability):")
display(summary)
dist_effect = pd.DataFrame({"poverty: mean SHAP of district": pov_shap["district"].groupby(hold["district_name"]).mean(),
                            "vup: mean SHAP of district": vup_shap["district"].groupby(hn["district_name"]).mean()}).sort_values("vup: mean SHAP of district")
dist_effect.to_csv(OUT / "shap_district_effects.csv")
print("District contributions (log-odds), sorted by the VUP model:")
display(dist_effect.head(8)); display(dist_effect.tail(5))
liv_effect = pd.DataFrame({"poverty": pov_shap["main_livelihood_pf"].groupby(hold["main_livelihood_pf"]).mean(),
                           "vup": vup_shap["main_livelihood_pf"].groupby(hn["main_livelihood_pf"]).mean()})
display(liv_effect)

## 3. Local explanations: 10 high-need, uncovered households

**Selection rule (fixed in advance).**
1. Start from holdout households that are high-need, receive no VUP component, and have a predicted poverty risk of at least 0.5.
2. Take the 10 with the **lowest predicted VUP coverage**: the most underserved profiles among clearly vulnerable, uncovered households.

For each household: the six largest SHAP contributions in each model. In both models, red bars raise the predicted probability and blue bars lower it. Households are identified by case number, district and area; survey IDs are only in the CSV.

In [ ]:
pov_scores = pd.read_parquet(pm.SCORES_PATH)[["hhid", "poverty_score"]]
cand = hn[hn["split"].eq("holdout") & hn["vup_any"].eq(0) & hn["poverty_score"].ge(0.5)]
cases = cand.nsmallest(10, "predicted_vup_coverage").reset_index(drop=True)
cases["case"] = [f"Case {i}" for i in range(1, 11)]
print(f"Eligible households: {len(cand)} of {int((hn['split'].eq('holdout') & hn['vup_any'].eq(0)).sum())} uncovered high-need holdout households")
display(cases[["case", "district_name", "area", "poor", "poverty_score", "predicted_vup_coverage", "main_livelihood_pf", "hh_size", "head_female", "head_education_code", "n_key_assets", "price_change_pct"]])

hold_idx = hold.reset_index().set_index("hhid")["index"]
hn_idx = hn.reset_index().set_index("hhid")["index"]
loc_pov = it.local_table(pov_shap.loc[hold_idx[cases["hhid"]].to_numpy()], hold.loc[:, pov_spec.columns + ["district_name"]], pov_base)
loc_vup = it.local_table(vup_shap.loc[hn_idx[cases["hhid"]].to_numpy()], hn.loc[:, vup_spec.columns + ["district_name"]], vup_base)
loc_pov["model"], loc_vup["model"] = "poverty", "vup_coverage"
row_to_case_p = dict(zip(hold_idx[cases["hhid"]].to_numpy(), cases["case"]))
row_to_case_v = dict(zip(hn_idx[cases["hhid"]].to_numpy(), cases["case"]))
loc_pov["case"], loc_vup["case"] = loc_pov["row"].map(row_to_case_p), loc_vup["row"].map(row_to_case_v)
local = pd.concat([loc_pov, loc_vup]).merge(cases[["case", "hhid", "district_name", "area"]], on="case").drop(columns="row")
local.to_csv(OUT / "local_explanations_10_households.csv", index=False)
local.head(12)

In [ ]:
RED, BLUE = "#e34948", "#2a78d6"
def fmt_value(f, v):
    if f == "district":
        return ""
    if isinstance(v, str):
        return f" = {LIV.get(v, v)}"
    return f" = {v:.2f}" if isinstance(v, float) and not float(v).is_integer() else f" = {int(v)}" if pd.notna(v) else " = missing"

def local_figure(table, name, title, prob_label):
    fig, axes = plt.subplots(5, 2, figsize=(14, 17))
    fig.patch.set_facecolor(pl.SURFACE)
    fig.text(0.02, 0.982, title, fontsize=14, fontweight="bold", color=pl.INK["primary"])
    fig.text(0.02, 0.968, "Six largest SHAP contributions (log-odds). Red raises the predicted probability, blue lowers it.", fontsize=10, color=pl.INK["secondary"])
    fig.text(0.02, 0.956, "Holdout households with high need, no VUP and predicted poverty risk >= 0.5", fontsize=10, color=pl.INK["secondary"])
    fig.text(0.02, 0.005, pl.SOURCE, fontsize=8, color=pl.INK["muted"])
    for ax, (case, g) in zip(axes.ravel(), table.groupby("case", sort=False)):
        pl.style(ax)
        g = g.iloc[::-1]
        ax.barh(range(len(g)), g["shap_log_odds"], color=[RED if v > 0 else BLUE for v in g["shap_log_odds"]], height=0.65)
        labels = [f"{it.pretty(f)}{fmt_value(f, v)}" for f, v in zip(g["feature"], g["value"])]
        ax.set_yticks(range(len(g)), labels, fontsize=8.5)
        ax.axvline(0, color=pl.BASELINE, linewidth=1)
        r = g.iloc[0]
        ax.set_title(f"{case}: {r['district_name']}, {r['area']}. {prob_label} {r['predicted_probability']:.2f}", fontsize=9.5, color=pl.INK["primary"], loc="left")
        ax.tick_params(axis="x", labelsize=8)
    fig.subplots_adjust(top=0.935, bottom=0.03, left=0.2, right=0.98, hspace=0.45, wspace=0.85)
    path = OUT / f"local_explanations_{name}.png"
    fig.savefig(path, dpi=150, facecolor=pl.SURFACE)
    plt.close(fig)
    return path

order = cases["case"].tolist()
lp = local[local["model"].eq("poverty")].set_index("case").loc[order].reset_index()
lv = local[local["model"].eq("vup_coverage")].set_index("case").loc[order].reset_index()
display(Image(str(local_figure(lp, "poverty", "Why the poverty model rates these 10 uncovered households as high risk", "predicted poverty risk"))))
display(Image(str(local_figure(lv, "vup", "Why the coverage model expects these 10 households to be rarely covered by VUP", "predicted VUP coverage"))))

In [ ]:
common_neg = lv[lv["shap_log_odds"] < 0]["feature"].value_counts().head(8)
common_pos = lp[lp["shap_log_odds"] > 0]["feature"].value_counts().head(8)
print("Most frequent drivers lowering predicted VUP coverage across the 10 cases:"); display(common_neg)
print("Most frequent drivers raising predicted poverty risk across the 10 cases:"); display(common_pos)

## 4. Subgroup fairness

**Poverty model (holdout).** For each group, at a single **national** cut-off (the top 20% of households by predicted risk):
- **Recall (people):** the share of the group's poor people selected (equal opportunity).
- **False positive rate:** the share of the group's non-poor households selected.
- **Calibration gap:** mean predicted minus observed poverty rate.

**VUP coverage model (all high-need households, out-of-sample predictions).** The calibration gap shows whether the model systematically over- or under-predicts coverage for a group. If it does, the "underserved" reading would be biased for that group.

Intervals come from a cluster bootstrap with 200 draws. Small groups have wide intervals and should not be over-read.

In [ ]:
def add_groups(d):
    d = d.copy()
    d["head_sex"] = np.where(d["head_female"].eq(1), "female head", "male head")
    d["head_age_group"] = pd.cut(d["head_age"], [0, 34, 59, 200], labels=["head under 35", "head 35-59", "head 60+"]).astype(str)
    d["disability"] = np.where(d["n_members_with_disability"].gt(0), "member with disability", "no disability")
    d["income_source"] = d["main_livelihood_pf"].map(LIV)
    d["province"] = d["province_name"].str.replace(" Province", "")
    return d

hp = add_groups(hold).assign(pov_pred=pov_pipe.predict_proba(hold[pov_spec.columns])[:, 1])
dims = ["province", "area", "income_source", "head_sex", "head_age_group", "disability"]
fair_pov = pd.concat({d: cm.subgroup_bootstrap(hp, "pov_pred", "poor", d, B=200,
                                              metrics=("roc_auc", "recall_people_at_national_top20", "calibration_gap", "false_positive_rate_at_national_top20"))
                      for d in dims}, names=["dimension", "group"])
fair_pov.to_csv(OUT / "fairness_poverty_model.csv")
fair_pov[["households", "positives", "observed_rate", "mean_predicted", "selected_share", "recall_people_at_national_top20",
          "recall_people_at_national_top20_ci_low", "recall_people_at_national_top20_ci_high", "false_positive_rate_at_national_top20",
          "calibration_gap", "calibration_gap_ci_low", "calibration_gap_ci_high", "roc_auc"]]

In [ ]:
hv = add_groups(hn).assign(poverty_status=lambda d: np.where(d["poor"].eq(1), "poor", "non-poor"))
fair_vup = pd.concat({d: cm.subgroup_bootstrap(hv, "predicted_vup_coverage", "vup_any", d, B=200, metrics=("roc_auc", "calibration_gap"))
                      for d in dims + ["poverty_status"]}, names=["dimension", "group"])
fair_vup.to_csv(OUT / "fairness_vup_model.csv")
fair_vup[["households", "positives", "observed_rate", "mean_predicted", "calibration_gap", "calibration_gap_ci_low", "calibration_gap_ci_high",
          "roc_auc", "roc_auc_ci_low", "roc_auc_ci_high"]]

In [ ]:
def fairness_figure():
    t = fair_pov.reset_index()
    t["label"] = t["group"]
    n = len(t)
    fig, (a1, a2) = pl.figure("Poverty model fairness on the holdout: who does a national top-20% ranking reach?",
                              "Left: share of each group's poor people selected. Right: calibration gap (predicted minus observed poverty rate). 95% cluster-bootstrap intervals",
                              width=12, height=1.9 + 0.3 * n, ncols=2)
    for ax, col, scale, ref, fmt in [(a1, "recall_people_at_national_top20", 100, None, "{:.0f}"), (a2, "calibration_gap", 100, 0, "{:+.1f}")]:
        tab = pd.DataFrame({"mean": t[col].to_numpy(), "ci_low": t[f"{col}_ci_low"].to_numpy(), "ci_high": t[f"{col}_ci_high"].to_numpy(),
                            "households": t["households"].to_numpy()}, index=[f"{d.replace('_', ' ')}: {g}" for d, g in zip(t["dimension"], t["group"])])
        pl.dot_ci(ax, tab, scale, pl.SERIES[2], label_fmt=fmt, reference=ref, show_n=(ax is a1))
        if ax is a2:
            ax.set_yticks(ax.get_yticks(), [""] * n)
    overall = cm.evaluate(hold["poor"], hp["pov_pred"], hold["weight"], hold["pop_wt"])["recall_people_top20"]
    a1.axvline(100 * overall, color=pl.INK["muted"], linestyle="--", linewidth=1)
    a1.set_title("Poor people selected, %  (dashed = national)", fontsize=10, color=pl.INK["secondary"], loc="left")
    a2.set_title("Calibration gap, percentage points", fontsize=10, color=pl.INK["secondary"], loc="left")
    return pl.save(fig, OUT / "fairness_poverty_model.png", top=1 - 1.3 / (1.9 + 0.3 * n), bottom=0.55 / (1.9 + 0.3 * n), left=0.27, wspace=0.08)
display(Image(str(fairness_figure())))

## 5. Model limitations

1. **Poverty is measured once, in 2023/24.** The model ranks households by risk of being poor at the time of the survey. It does not predict who is poor in 2026, and it does not apply the CPI shock to the poverty line.
2. **Consumption-derived features.** Expenditure shares, the CPI shock and the distress score come from the consumption module that defines poverty. They would not exist for households outside the survey. Without them (the PMT-feasible set), the holdout ROC-AUC falls from 0.90 to about 0.87–0.88 (notebook 09).
3. **Post-transfer consumption.** Poverty is measured after VUP and other transfers. Recipients can look less poor because of the transfer, so the poverty model may understate their pre-transfer need, and the coverage model inherits this.
4. **The coverage model is weak and descriptive.** Its holdout ROC-AUC is about 0.72, against 0.60 for the district baseline, with wide intervals (67 recipients in the holdout).
   - Most variation in who receives VUP is not explained by these household characteristics.
   - VUP's own rules are not observed: geographic phasing by sector, socio-economic categorisation, labour capacity for Public Works, and pregnancy or a child under two for NSDS.
   - "Underserved" therefore partly reflects program design (for example, NSDS needs young children), not only gaps.
5. **Probabilities.**
   - The poverty model's probabilities are close to calibrated (slope about 0.94).
   - The coverage model's probabilities are too spread out (slope about 0.5–0.7). Use them for ranking, not as exact rates.
6. **No tuning yet.** All models use library defaults. Tuning comes only after the baseline comparison, and must use the same cluster folds, never the holdout.
7. **SHAP is not causal.** It attributes the model's prediction among correlated inputs. Feature-level importance within the collinear groups (household size, employment counts, share/shock pairs) is unstable; group-level importance is more reliable.
8. **Small groups.** District effects and small subgroups (households with no employed member, Kigali high-need households) rest on few observations. Their intervals are wide.
9. **Program-free features.** VUP-sector jobs and VUP loans are removed from the features. Other indirect traces of program participation cannot be ruled out entirely, for example assets bought with transfers.

In [ ]:
print("Saved:", ", ".join(sorted(p.name for p in OUT.glob("*"))))